In [1]:
## Benchmark Forecasting

# We evaluate benchmark models separately for each store.
# Each store uses its own history to create 42-day forecasts, and the metrics are stored per store and per model.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pmdarima import auto_arima
except ImportError:
    auto_arima = None

# ==========================================================
# Load data
# ==========================================================
processed_dir = Path("data/processed")
if not processed_dir.exists():
    processed_dir = Path.cwd().parent / "data" / "processed"

sales = pd.read_csv(processed_dir / "sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales = sales.sort_values(["store_id", "date"])

# ==========================================================
# Helper functions
# ==========================================================

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def build_forecasts(train_series, test_len):
    forecasts = {}

    train_series = pd.Series(train_series).astype(float)
    train_series = train_series.replace([np.inf, -np.inf], np.nan).dropna()

    if len(train_series) < 10:
        return {
            "Mean": np.repeat(np.nan, test_len),
            "Naive": np.repeat(np.nan, test_len),
            "Drift": np.repeat(np.nan, test_len),
            "Seasonal Naive": np.repeat(np.nan, test_len),
            "AutoARIMA": np.repeat(np.nan, test_len),
        }

    forecasts["Mean"] = np.full(test_len, train_series.mean())
    forecasts["Naive"] = np.full(test_len, train_series.iloc[-1])

    drift = np.arange(1, test_len + 1)
    forecasts["Drift"] = train_series.iloc[-1] + drift * (
        (train_series.iloc[-1] - train_series.iloc[0]) / (len(train_series) - 1)
    )

    forecasts["Seasonal Naive"] = np.array([
        train_series.iloc[-7 + (i % 7)] for i in range(test_len)
    ])

    if auto_arima is not None:
        try:
            model = auto_arima(
                train_series,
                seasonal=True,
                m=7,
                stepwise=True,
                suppress_warnings=True,
                error_action="ignore",
            )
            forecasts["AutoARIMA"] = model.predict(n_periods=test_len)
        except Exception:
            forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)
    else:
        forecasts["AutoARIMA"] = np.repeat(np.nan, test_len)

    return forecasts

# ==========================================================
# Train / test split by store
# ==========================================================
# We use all rows before the 42-day holdout period as training.
# The forecast origin is the day before the first test date, so no extra day is removed.

forecast_horizon = 42
results = []

for store_id, store_df in sales.groupby("store_id"):
    store_df = store_df.sort_values("date").copy()
    store_df["sales"] = pd.to_numeric(store_df["sales"], errors="coerce")
    store_df = store_df.dropna(subset=["sales"])

    if len(store_df) < forecast_horizon + 5:
        continue

    train_series = store_df.iloc[:-forecast_horizon]["sales"].astype(float)
    test_series = store_df.iloc[-forecast_horizon:]["sales"].astype(float)
    test_dates = store_df.iloc[-forecast_horizon:]["date"]

    forecasts = build_forecasts(train_series, forecast_horizon)

    for model_name, pred in forecasts.items():
        pred = np.asarray(pred, dtype=float)
        if np.isnan(pred).any():
            continue

        results.append({
            "store_id": store_id,
            "model": model_name,
            "mae": mae(test_series.values, pred),
            "mape": mape(test_series.values, pred),
            "forecast_dates": json.dumps([d.strftime("%Y-%m-%d") for d in test_dates]),
            "actual_values": json.dumps([float(v) for v in test_series.values]),
            "forecast_values": json.dumps([float(v) for v in pred]),
        })

results_df = pd.DataFrame(results)
print(results_df.head())
print(f"\nTotal rows: {len(results_df)}")

# ==========================================================
# Save results
# ==========================================================
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "benchmark_results_per_store.csv"
results_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

# ==========================================================
# Summary by model
# ==========================================================
summary_df = (
    results_df.groupby("model")[["mae", "mape"]]
    .mean()
    .sort_values("mae")
)
print(summary_df)


   store_id           model          mae        mape  \
0   store_1            Mean  1062.011176   12.494584   
1   store_1           Naive  3696.261905  100.000000   
2   store_1           Drift  3871.385628  103.971858   
3   store_1  Seasonal Naive  1383.833333   39.491304   
4  store_10            Mean  1937.339974   16.214092   

                                      forecast_dates  \
0  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
1  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
2  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
3  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   
4  ["2015-06-08", "2015-06-09", "2015-06-10", "20...   

                                       actual_values  \
0  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
1  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
2  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
3  [4071.0, 4102.0, 3591.0, 3627.0, 3695.0, 4256....   
4  [6177.0, 5109.0, 5569.0, 5911.0, 5757.0, 57

In [3]:
#The poor performance of the Naive and Drift baselines is mainly due to the data pattern around the test split. 
# The last training point is unusually low, while the first test point rises sharply, 
# so these simple methods struggle to adapt and produce large errors. 
# This is a characteristic of the series rather than a coding issue.

In [3]:
# ==========================================================
# Mean Benchmark Rolling Forecast Origin Cross Validation
# ==========================================================

print("=" * 60)
print("MEAN BENCHMARK ROLLING FORECAST ORIGIN CV")
print("=" * 60)

# Number of folds
n_folds = 3

# Forecast horizon
forecast_horizon = 42

cv_results = []

for fold in range(n_folds):

    print(f"\nRunning Fold {fold + 1}/{n_folds}")

    fold_mae = []
    fold_mape = []

    # Evaluate each store separately
    for store_id, store_df in sales.groupby("store_id"):

        store_df = (
            store_df
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(store_df) < forecast_horizon * (fold + 1) + 10:
            continue

        # Same rolling split as Random Forest
        valid_end = len(store_df) - fold * forecast_horizon
        valid_start = valid_end - forecast_horizon

        train_df = store_df.iloc[:valid_start]
        valid_df = store_df.iloc[valid_start:valid_end]

        train_series = train_df["sales"].astype(float)
        valid_series = valid_df["sales"].astype(float)

        # Mean Forecast
        pred = np.full(
            len(valid_series),
            train_series.mean()
        )

        # Metrics
        fold_mae.append(
            mae(valid_series.values, pred)
        )

        fold_mape.append(
            mape(valid_series.values, pred)
        )

    cv_results.append({

        "Fold": fold + 1,

        "MAE": np.mean(fold_mae),

        "MAPE": np.mean(fold_mape)

    })

cv_results = pd.DataFrame(cv_results)

print("\n")
print("=" * 60)
print("MEAN CROSS VALIDATION RESULTS")
print("=" * 60)

print(cv_results)

print("\nAverage Performance")

print(f"MAE  : {cv_results['MAE'].mean():.2f}")
print(f"MAPE : {cv_results['MAPE'].mean():.2f}%")

MEAN BENCHMARK ROLLING FORECAST ORIGIN CV

Running Fold 1/3

Running Fold 2/3

Running Fold 3/3


MEAN CROSS VALIDATION RESULTS
   Fold          MAE       MAPE
0     1  2126.522666  20.747651
1     2  2947.349507  26.591152
2     3  2436.801513  22.284249

Average Performance
MAE  : 2503.56
MAPE : 23.21%


In [4]:
from pathlib import Path

raw_dir = Path("data/raw")

metadata = pd.read_csv(
    raw_dir / "metadata.csv"
)

benchmark = pd.read_csv(
    processed_dir / "benchmark_results_per_store.csv"
)

benchmark = benchmark.merge(
    metadata[
        ["store_id", "store_type"]
    ],
    on="store_id",
    how="left"
)

In [5]:
mean_by_type = (

    benchmark[
        benchmark["model"]=="Mean"
    ]

    .groupby("store_type")

    [["mae","mape"]]

    .mean()

)

print(mean_by_type)

                    mae       mape
store_type                        
a           2153.439939  21.353938
b           2054.606635  17.450511
c           2001.523516  19.878449
d           2126.608114  20.150884


In [6]:
compare = (
    benchmark
    .groupby(["store_type", "model"])[["mae", "mape"]]
    .mean()
    .round(2)
)

print(compare)

                               mae    mape
store_type model                          
a          Drift           6111.14  102.08
           Mean            2153.44   21.35
           Naive           5868.41   98.40
           Seasonal Naive  2111.26   38.44
b          Drift           3654.64   33.31
           Mean            2054.61   17.45
           Naive           3525.05   32.03
           Seasonal Naive  2056.71   18.32
c          Drift           5804.43  103.38
           Mean            2001.52   19.88
           Naive           5585.86  100.00
           Seasonal Naive  1965.30   39.82
d          Drift           6074.50  102.49
           Mean            2126.61   20.15
           Naive           5856.66   99.28
           Seasonal Naive  2278.55   43.15
